# Generate parish comparison reports

Create one Markdown report for **every target parish** in the clustering target summary. Each report includes parish information, current recorded ministries, descriptions of the ranked neighbors, and neighbor ministry labels not currently identified for the target. Target selection follows `data/05_cluster_parishes/6_run_metadata.json`; no diocese name is hard-coded.

Read existing pipeline artifacts only. No scraping or LLM calls are made. Final reports belong in `reports/parish_comparisons/`; intermediate comparison datasets and the manifest belong in `data/06_generate_reports/`. Run from the project root after completing notebooks 03?05.


## 0 Settings and inputs

Use the current clustering assignments, target summary, and nearest matches together. Parish information comes from the clustering snapshot; ministry information comes from the latest page-level extraction results. Identity checks use parish identifier, name, diocese, and site URL together, so changed row indices cannot silently attach ministries to another parish.

Reports use Markdown so each parish has a readable, portable file with clickable source evidence. Filenames use parish identifiers plus sanitized names to distinguish repeated parish names.


In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

from utils.parish_reports import (
    IDENTITY_COLUMNS, generate_parish_reports, ministry_coverage,
    neighbor_ministry_gaps, summarize_ministries, validate_parish_rows,
)

CLUSTER_DIR = Path("data/05_cluster_parishes")
MINISTRY_DIR = Path("data/03_extract_ministries")
OUTPUT_DIR = Path("data/06_generate_reports")
REPORT_DIR = Path("reports/parish_comparisons")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_paths = {
    "parishes": CLUSTER_DIR / "4_parish_clusters.csv",
    "neighbors": CLUSTER_DIR / "5_target_nearest_matches.csv",
    "target_summary": CLUSTER_DIR / "5_target_summary.csv",
    "cluster_metadata": CLUSTER_DIR / "6_run_metadata.json",
    "ministries": MINISTRY_DIR / "parish_ministries.csv",
    "listed_pages": MINISTRY_DIR / "1_pages_with_rules.csv",
}
for path in input_paths.values():
    if not path.is_file():
        raise FileNotFoundError(f"Required pipeline input is missing: {path}")
cluster_metadata = json.loads(input_paths["cluster_metadata"].read_text(encoding="utf-8"))
TARGET_DIOCESE = cluster_metadata["target_diocese"]
print(f"Target diocese: {TARGET_DIOCESE}")


Target diocese: Diocese of Fort Wayne-South Bend


## 1 Load and validate parish identities

Keep every target, including those with missing ministries or no neighbors. Check the neighbor identities against the parish snapshot and retain extraction rows only when both their parish identifier and identity agree. Export unmatched ministry/page rows for review; exclude those rows from comparisons rather than guessing a parish association.

The assigned cluster is a similarity grouping, not a rating. Neighbor distances use the transformed KMeans feature space, not geographic mileage. Preserve provisional/imputed assignment information in reports.


In [2]:
parishes_df = pd.read_csv(input_paths["parishes"], keep_default_na=False, dtype={"county_fips": "string"})
nearest_matches_df = pd.read_csv(input_paths["neighbors"], keep_default_na=False)
target_summary_df = pd.read_csv(input_paths["target_summary"], keep_default_na=False)
assert parishes_df["parish_index"].is_unique
assert target_summary_df["target_parish_index"].is_unique
assert set(target_summary_df["target_parish_index"]) == set(
    parishes_df.loc[parishes_df["diocese"].eq(TARGET_DIOCESE), "parish_index"]
), "Target summary and configured diocese do not agree"
for prefix in ["target", "peer"]:
    identities = nearest_matches_df[[f"{prefix}_{column}" for column in IDENTITY_COLUMNS]].rename(
        columns={f"{prefix}_{column}": column for column in IDENTITY_COLUMNS}
    )
    _, unmatched = validate_parish_rows(identities, parishes_df)
    assert unmatched.empty, f"Stale or unknown {prefix} identities in nearest matches"

ministry_columns = IDENTITY_COLUMNS + ["page_url", "ministry", "category", "prediction_status"]
ministry_results_df = pd.read_csv(input_paths["ministries"], usecols=ministry_columns, keep_default_na=False)
listed_pages_df = pd.read_csv(input_paths["listed_pages"], usecols=IDENTITY_COLUMNS + ["page_url"], keep_default_na=False)
ministry_results_df, unmatched_ministries_df = validate_parish_rows(ministry_results_df, parishes_df)
listed_pages_df, unmatched_pages_df = validate_parish_rows(listed_pages_df, parishes_df)
unmatched_ministries_df.to_csv(OUTPUT_DIR / "1_unmatched_ministry_rows.csv", index=False)
unmatched_pages_df.to_csv(OUTPUT_DIR / "1_unmatched_page_rows.csv", index=False)
validation_df = pd.DataFrame([
    {"input": "ministry_results", "matched_rows": len(ministry_results_df), "unmatched_rows": len(unmatched_ministries_df)},
    {"input": "listed_pages", "matched_rows": len(listed_pages_df), "unmatched_rows": len(unmatched_pages_df)},
])
validation_df.to_csv(OUTPUT_DIR / "1_input_validation.csv", index=False)
print(f"Preparing reports for {len(target_summary_df)} targets and {len(nearest_matches_df)} neighbor matches")
validation_df


Preparing reports for 80 targets and 800 neighbor matches


,input,matched_rows,unmatched_rows
0,ministry_results,115296,0
1,listed_pages,115296,0


## 2 Build parish ministry lists and coverage

Accept nonempty ministry labels only from successful (`ok`) or rule-derived (`rule`) results. Deduplicate labels per parish after normalizing Unicode, case, punctuation, and spacing. Preserve the displayed ministry names, categories, and every source page URL. Do not infer synonyms or combine different ministry names automatically.

Coverage distinguishes no discovered pages, no extraction results, partial processing, and listed pages resolved or skipped by a rule. A successful prediction with no ministry label is still a processed page. A parish with no recorded labels may still operate ministries; the report states this explicitly. Save the normalized parish ministry list and coverage table.


In [3]:
parish_ministries_df = summarize_ministries(ministry_results_df)
ministry_coverage_df = ministry_coverage(parishes_df, listed_pages_df, ministry_results_df, parish_ministries_df)
parish_ministries_df.to_csv(OUTPUT_DIR / "2_parish_ministries.csv", index=False)
ministry_coverage_df.to_csv(OUTPUT_DIR / "2_ministry_coverage.csv", index=False)
print(f"Identified {len(parish_ministries_df)} distinct parish/ministry pairs")
ministry_coverage_df.loc[
    ministry_coverage_df["parish_index"].isin(target_summary_df["target_parish_index"]),
    ["parish_index", "name", "n_ministries", "n_listed_pages", "n_unresolved_listed_pages", "coverage_status"],
].head(20)


Identified 4692 distinct parish/ministry pairs


,parish_index,name,n_ministries,n_listed_pages,n_unresolved_listed_pages,coverage_status
784,784,Blessed Sacrament,2,12,0,resolved_or_rule_skipped
785,785,Cathedral of the Immaculate Conception,5,41,0,resolved_or_rule_skipped
786,786,Christ the King,0,0,0,no_listed_pages
787,787,Corpus Christi,6,55,0,resolved_or_rule_skipped
788,788,Holy Cross,20,177,0,resolved_or_rule_skipped
789,789,Holy Family,7,52,0,resolved_or_rule_skipped
790,790,"Immaculate Conception, Auburn",5,35,0,resolved_or_rule_skipped
791,791,"Immaculate Conception, Kendallville",0,0,0,no_listed_pages
792,792,"Immaculate Conception, LaOtto",1,42,0,resolved_or_rule_skipped
793,793,Most Precious Blood,9,822,0,resolved_or_rule_skipped


## 3 Compare target and neighbor ministries

For each target, compare every ranked neighbor's normalized ministry labels against the target's labels. Keep one observation per target, neighbor, and additional ministry. Reports combine these observations by ministry and rank them by the number of distinct supporting neighbors, then alphabetically. Counts represent neighbor parishes, not duplicate source pages.

Save `3_neighbor_ministry_gaps.csv` with target and neighbor identities, neighbor rank/distance, ministry/category, and all source URLs. The reports call these ?not currently identified? labels: extraction gaps, broad or synonymous labels, and incomplete evidence can produce apparent differences. No new ministry recommendation is generated by an LLM.


In [4]:
neighbor_gaps_df = neighbor_ministry_gaps(nearest_matches_df, parish_ministries_df)
gap_columns = [
    "target_parish_index", "target_name", "peer_parish_index", "peer_name", "peer_site_url",
    "rank", "distance", "cluster", "ministry_key", "ministry", "category", "n_source_pages", "source_urls",
]
neighbor_gaps_df[gap_columns].to_csv(OUTPUT_DIR / "3_neighbor_ministry_gaps.csv", index=False)
assert not neighbor_gaps_df.duplicated(["target_parish_index", "peer_parish_index", "ministry_key"]).any()
print(f"Found {len(neighbor_gaps_df)} target/neighbor ministry differences")
neighbor_gaps_df.groupby("ministry")["target_parish_index"].nunique().sort_values(ascending=False).head(15)


Found 5824 target/neighbor ministry differences


ministry
Knights of Columbus                              58
Prayer Ministry                                  57
Sacramental Preparation                          57
Rite of Christian Initiation of Adults (RCIA)    56
Eucharistic Adoration                            55
Music Ministry                                   55
Liturgy and Worship                              54
Faith Formation                                  53
Altar Servers                                    52
Youth Ministry                                   52
Prayer and Spirituality                          51
Anointing of the Sick                            49
Adult Faith Formation                            48
Lectors                                          47
Marriage Preparation                             46
Name: target_parish_index, dtype: int64

## 4 Write reports and the index

Generate one report for every target summary row. Each report contains:

1. Parish identity, address, website, county/digital/school profile, cluster, and assignment quality.
2. Current recorded ministries with categories, source links, and extraction coverage.
3. Each ranked neighbor's parish description, original feature values, distance, and ministry coverage.
4. Additional ministry labels found among neighbors, with supporting neighbor counts and clickable evidence.

Source links in the current-ministry table show up to three pages for readability; all URLs remain in `2_parish_ministries.csv` and `3_neighbor_ministry_gaps.csv`. Neighbor ministries are supported by links to their source pages. Reports with missing extraction data or provisional clustering assignments explain those limits.

Write final reports under `reports/parish_comparisons/` and an `index.md` linking every report. Reruns replace reports for the current targets and the index; other existing report files are not removed automatically. Save the report manifest and source hashes under `data/06_generate_reports/`.


In [5]:
report_manifest_df = generate_parish_reports(
    parishes_df, target_summary_df, nearest_matches_df,
    parish_ministries_df, ministry_coverage_df, neighbor_gaps_df,
    report_dir=REPORT_DIR, target_diocese=TARGET_DIOCESE,
)
assert len(report_manifest_df) == len(target_summary_df)
assert report_manifest_df["report_path"].is_unique
assert report_manifest_df["report_path"].map(lambda name: Path(name).is_file()).all()
report_manifest_df.to_csv(OUTPUT_DIR / "4_report_manifest.csv", index=False)
run_metadata = {
    "target_diocese": TARGET_DIOCESE,
    "n_reports": len(report_manifest_df), "n_neighbor_matches": len(nearest_matches_df),
    "n_targets_without_ministry_labels": int(report_manifest_df["n_current_ministries"].eq(0).sum()),
    "n_provisional_targets": int(report_manifest_df["assignment_quality"].eq("provisional_imputed").sum()),
    "report_directory": REPORT_DIR.as_posix(), "report_index": (REPORT_DIR / "index.md").as_posix(),
    "ministry_matching": "Unicode/case/punctuation/spacing normalization; no synonym inference",
    "accepted_prediction_statuses": ["ok", "rule"],
    "unmatched_ministry_rows": len(unmatched_ministries_df), "unmatched_listed_page_rows": len(unmatched_pages_df),
    "inputs": {name: {"path": path.as_posix(), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for name, path in input_paths.items()},
}
(OUTPUT_DIR / "4_run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
print(f"Generated {len(report_manifest_df)} parish reports")
print(f"Open the report index: {REPORT_DIR / 'index.md'}")
report_manifest_df.head(20)


Generated 80 parish reports
Open the report index: reports\parish_comparisons\index.md


,target_parish_index,target_name,target_diocese,report_path,n_current_ministries,n_neighbors,n_additional_ministry_labels,ministry_coverage_status,assignment_quality
0,784,Blessed Sacrament,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0784_blessed-sacram...,2,10,55,resolved_or_rule_skipped,complete
1,785,Cathedral of the Immaculate Conception,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0785_cathedral-of-t...,5,10,30,resolved_or_rule_skipped,complete
2,786,Christ the King,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0786_christ-the-kin...,0,10,56,no_listed_pages,provisional_imputed
3,787,Corpus Christi,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0787_corpus-christi.md,6,10,104,resolved_or_rule_skipped,complete
4,788,Holy Cross,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0788_holy-cross.md,20,10,45,resolved_or_rule_skipped,complete
5,789,Holy Family,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0789_holy-family.md,7,10,91,resolved_or_rule_skipped,complete
6,790,"Immaculate Conception, Auburn",Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0790_immaculate-con...,5,10,51,resolved_or_rule_skipped,complete
7,791,"Immaculate Conception, Kendallville",Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0791_immaculate-con...,0,10,51,no_listed_pages,provisional_imputed
8,792,"Immaculate Conception, LaOtto",Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0792_immaculate-con...,1,10,6,resolved_or_rule_skipped,complete
9,793,Most Precious Blood,Diocese of Fort Wayne-South Bend,reports/parish_comparisons/0793_most-precious-...,9,10,40,resolved_or_rule_skipped,complete


## 5 Review and continue

Open `reports/parish_comparisons/index.md` and inspect individual reports. Review targets with no ministry labels, partial extraction, or imputed cluster assignments before treating listed differences as planning opportunities. The manifest records ministry counts and coverage per target, and the comparison CSV preserves the full evidence behind each apparent difference.

Reports describe the pipeline's recorded information, including AI predictions. Confirm parish activities against the cited pages or parish staff before interpreting an unrecorded ministry as absent.
